# NFL Game Simulator v6

**Files in one folder:** `nfl_game_sim_v6.py`, `nfl_props_v6.py`, `nfl_weekly_v6.py`, `fetch_dk_props.py`, `run_week.py`, this notebook, and (optional) `odds_api_key.txt` with your Odds API key. After replacing any `.py` file: **Kernel → Restart**.

**What's new in v6**
- **One command per week.** `weekly_report()` refreshes this season's data, simulates every game and every player, pulls DraftKings' lines, logs the predictions, and saves `reports/<season>_wk<week>.xlsx` and `.html`. From a terminal: `python run_week.py --week 5`.
- **Prediction log + closing line value.** Every run logs predictions with the market line at that moment (`logs/`). `grade_log()` grades them after the games, including whether the market moved toward the model's number before kickoff.
- **Consistent props.** In every simulated game, the QB's completions, yards, and TDs are the sum of what his receivers did, and yards rise and fall with that game's score. This makes same-game combos priceable (section 4).
- **Scoring level.** Tested, and not applied by default: it made over/under picks worse in backtests. The dashboard reports the league's scoring level for context.

Each cell below depends only on `report` (section 1) or `g` (section 2), so they can be re-run in any order after those two.

In [1]:
# First time only
# %pip install pandas numpy pyarrow openpyxl matplotlib

In [2]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
from nfl_weekly_v6 import *          # also gives you ns (game model) and props (props model)
pd.set_option("display.max_rows", 200)

## 1. Run the week
Re-run this cell whenever you want fresh data or lines: after Wednesday's first injury report, after Friday's final report, and Sunday morning after inactives. Every run is logged, so the log keeps the latest pre-kickoff prediction.

In [3]:
SEASON, WEEK = 2026, 8
REFRESH = True        # re-download this season's data and pull ESPN's live injury list
REFRESH_HOURS = 12    # files older than this re-download; 0 = force everything fresh (IR news, roster moves)
WEATHER = {}          # e.g. {"BUF": {"temp": 31, "wind": 22}}  (keyed by home team)
REFEREES = {}         # e.g. {"PHI": "Shawn Hochuli"}          (keyed by home team)
N_SIMS = 40000        # simulations per game for the full week (more = smoother, slower)
LINES = "auto"        # "auto" = Odds API if you have a key, else FanDuel, else ESPN | "fanduel" | "espn" | "odds_api"

report = weekly_report(week=WEEK, season=SEASON, refresh=REFRESH, refresh_hours=REFRESH_HOURS,
                       weather=WEATHER, referees=REFEREES, n_sims=N_SIMS, lines=LINES)

Data is fresh (no download needed).
No odds_api lines; trying fanduel.
  CAR @ GB: 94 FanDuel prices
  TEN @ CIN: 0 FanDuel prices
  BAL @ BUF: 108 FanDuel prices
  ARI @ DAL: 0 FanDuel prices
  CLE @ PIT: 0 FanDuel prices
  LAC @ LA: 0 FanDuel prices
  IND @ JAX: 42 FanDuel prices
  MIN @ DET: 62 FanDuel prices
  ATL @ TB: 0 FanDuel prices
  LV @ NYJ: 0 FanDuel prices
  KC @ DEN: 62 FanDuel prices
  NE @ MIA: 84 FanDuel prices
  CHI @ SEA: 49 FanDuel prices
  PHI @ WAS: 0 FanDuel prices
Wrote 501 FanDuel prices for 14 games to fd_props_current.csv

2026 week 8 | Week 8 injury report; live ESPN injury list as of Wed Oct 28, 08:13 AM Arizona time has the final word
  matchup  home_win_pct  model_line  vegas_line  model_total  vegas_total
 CAR @ GB          60.4        -3.8        -3.5         47.1         47.5
BAL @ BUF          61.7        -4.1        -3.0         55.7         54.5
TEN @ CIN          68.1        -6.3       -10.0         43.1         46.5
ARI @ DAL          63.2        

In [4]:
games_table(report)

,Matchup,QBs (away / home),Home win %,Model line,Vegas line,Model's side,Spread gap,Model total,Vegas total,Model's total lean,Total gap,Final
0,CAR @ GB,B.Young / J.Love,60.4,-3.8,-3.5,GB -3.5,0.3,47.1,47.5,Under 47.5,0.4,
1,BAL @ BUF,T.Huntley (55%) / L.Jackson (45%) / J.Allen,61.7,-4.1,-3.0,BUF -3,1.1,55.7,54.5,Over 54.5,1.2,
2,TEN @ CIN,C.Ward / J.Burrow,68.1,-6.3,-10.0,TEN +10,3.7,43.1,46.5,Under 46.5,3.4,
3,ARI @ DAL,J.Brissett / D.Prescott,63.2,-4.8,-7.0,ARI +7,2.2,54.8,54.5,Over 54.5,0.3,
4,MIN @ DET,K.Murray / J.Goff,41.0,2.9,-3.0,MIN +3,5.9,44.9,46.5,Under 46.5,1.6,
5,IND @ JAX,D.Jones / T.Lawrence,65.2,-5.3,-9.5,IND +9.5,4.2,45.3,46.5,Under 46.5,1.2,
6,LV @ NYJ,K.Cousins / G.Smith,48.1,0.6,-1.5,LV +1.5,2.1,49.1,45.5,Over 45.5,3.6,
7,CLE @ PIT,D.Watson / A.Rodgers,52.7,-1.0,-3.5,CLE +3.5,2.5,42.8,39.5,Over 39.5,3.3,
8,ATL @ TB,M.Penix / B.Mayfield,52.6,-1.1,-2.5,ATL +2.5,1.4,46.3,45.5,Over 45.5,0.8,
9,LAC @ LA,J.Herbert / M.Stafford,51.7,-0.7,-6.5,LAC +6.5,5.8,47.2,45.5,Over 45.5,1.7,


In [5]:
import numpy as np

def games_table_graded(report):
    """games_table plus whether the model's side and total lean were right."""
    t = games_table(report)
    g = report["games"]
    sc = report["data"]["games"].set_index("game_id")
    hs, as_ = g.game_id.map(sc.home_score), g.game_id.map(sc.away_score)
    done = hs.notna() & as_.notna()

    # spread: the model takes the home team when its line is below Vegas (negative = home favored)
    cover = (hs - as_) + g.vegas_line            # > 0 home covered, < 0 away covered, 0 push
    model_home = g.model_line < g.vegas_line
    spread = np.where(~done | g.vegas_line.isna(), "",
             np.where(cover == 0, "Push",
             np.where((cover > 0) == model_home, "✅ Correct", "❌ Wrong")))

    # total: the model leans over when its total is above Vegas
    pts = hs + as_
    model_over = g.model_total > g.vegas_total
    total = np.where(~done | g.vegas_total.isna(), "",
            np.where(pts == g.vegas_total, "Push",
            np.where((pts > g.vegas_total) == model_over, "✅ Correct", "❌ Wrong")))

    t.insert(t.columns.get_loc("Model's side") + 1, "Side result", spread)
    t.insert(t.columns.get_loc("Model's total lean") + 1, "Total result", total)

    for col in ["Side result", "Total result"]:
        w, l, p = (t[col] == "✅ Correct").sum(), (t[col] == "❌ Wrong").sum(), (t[col] == "Push").sum()
        if w + l + p:
            print(f"{col}: {w}-{l}" + (f"-{p}" if p else "") + f"  ({w / max(w + l, 1):.0%})")
    return t

games_table_graded(report)

,Matchup,QBs (away / home),Home win %,Model line,Vegas line,Model's side,Side result,Spread gap,Model total,Vegas total,Model's total lean,Total result,Total gap,Final
0,CAR @ GB,B.Young / J.Love,60.4,-3.8,-3.5,GB -3.5,,0.3,47.1,47.5,Under 47.5,,0.4,
1,BAL @ BUF,T.Huntley (55%) / L.Jackson (45%) / J.Allen,61.7,-4.1,-3.0,BUF -3,,1.1,55.7,54.5,Over 54.5,,1.2,
2,TEN @ CIN,C.Ward / J.Burrow,68.1,-6.3,-10.0,TEN +10,,3.7,43.1,46.5,Under 46.5,,3.4,
3,ARI @ DAL,J.Brissett / D.Prescott,63.2,-4.8,-7.0,ARI +7,,2.2,54.8,54.5,Over 54.5,,0.3,
4,MIN @ DET,K.Murray / J.Goff,41.0,2.9,-3.0,MIN +3,,5.9,44.9,46.5,Under 46.5,,1.6,
5,IND @ JAX,D.Jones / T.Lawrence,65.2,-5.3,-9.5,IND +9.5,,4.2,45.3,46.5,Under 46.5,,1.2,
6,LV @ NYJ,K.Cousins / G.Smith,48.1,0.6,-1.5,LV +1.5,,2.1,49.1,45.5,Over 45.5,,3.6,
7,CLE @ PIT,D.Watson / A.Rodgers,52.7,-1.0,-3.5,CLE +3.5,,2.5,42.8,39.5,Over 39.5,,3.3,
8,ATL @ TB,M.Penix / B.Mayfield,52.6,-1.1,-2.5,ATL +2.5,,1.4,46.3,45.5,Over 45.5,,0.8,
9,LAC @ LA,J.Herbert / M.Stafford,51.7,-0.7,-6.5,LAC +6.5,,5.8,47.2,45.5,Over 45.5,,1.7,


In [6]:
def td_board(report, top=4):
    """Each game's top `top` anytime-TD favorites by sportsbook odds, with the model's take and EV."""
    dk = report.get("dk_yes")
    if dk is None or not len(dk):
        print("No anytime-TD prices in this report. Set LINES = 'fanduel' (or 'auto') in Section 1 and re-run it.")
        return None
    kick = report["games"].set_index("matchup").kickoff_utc
    rows = []
    for (h, a), (g, sp) in report["sims"].items():
        game = f"{a} @ {h}"
        teams = sp.means.set_index("player").team
        d = dk[(dk.home == h) & (dk.away == a)].sort_values("yes_price").drop_duplicates("key")
        n = 0
        for _, r in d.iterrows():
            name = sp._name(r.player)
            if (name, "anytime_td") not in sp.draws:
                continue          # team defenses / players the model doesn't project
            p = sp.prob_over(name, "anytime_td", None)
            n += 1
            k = kick.get(game)
            rows.append({"_kick": k, "Kickoff (AZ)": k.tz_convert("America/Phoenix").strftime("%a %I:%M %p"),
                         "Game": game, "Rank": n, "Player": name, "Team": teams.get(name, ""),
                         "Book odds": props._american(r.yes_price),
                         "Book %": round(100 / r.yes_price, 1),
                         "Model %": round(p * 100, 1),
                         "Model fair odds": props._american_from_prob(p),
                         "EV per $1": round(p * r.yes_price - 1, 3)})
            if n == top:
                break
    if not rows:
        print("No anytime-TD prices matched this week's games yet.")
        return None
    return pd.DataFrame(rows).sort_values(["_kick", "Game", "Rank"]).drop(columns="_kick").reset_index(drop=True)

TOP = 6
board = td_board(report, top=TOP)
if board is not None:
    board.to_csv(f"reports/{SEASON}_wk{WEEK:02d}_td_board.csv", index=False)
    display(board.style
            .format({"Book %": "{:.1f}", "Model %": "{:.1f}", "EV per $1": "{:+.2f}"})
            .background_gradient(subset=["EV per $1"], cmap="RdYlGn", vmin=-0.4, vmax=0.4)
            .hide(axis="index"))

Kickoff (AZ),Game,Rank,Player,Team,Book odds,Book %,Model %,Model fair odds,EV per $1
Thu 05:15 PM,CAR @ GB,1,Josh Jacobs,GB,-135,57.4,39.7,+152,-0.31
Thu 05:15 PM,CAR @ GB,2,Chuba Hubbard,CAR,-115,53.5,52.7,-111,-0.01
Thu 05:15 PM,CAR @ GB,3,Christian Watson,GB,+130,43.5,33.7,+197,-0.23
Thu 05:15 PM,CAR @ GB,4,Matthew Golden,GB,+160,38.5,26.2,+282,-0.32
Thu 05:15 PM,CAR @ GB,5,Tucker Kraft,GB,+170,37.0,25.4,+294,-0.32
Thu 05:15 PM,CAR @ GB,6,Tetairoa McMillan,CAR,+175,36.4,22.4,+345,-0.38
Sun 11:00 AM,BAL @ BUF,1,Derrick Henry,BAL,-240,70.6,73.0,-270,+0.03
Sun 11:00 AM,BAL @ BUF,2,Josh Allen,BUF,-170,63.0,38.4,+160,-0.39
Sun 11:00 AM,BAL @ BUF,3,James Cook,BUF,-165,62.3,61.1,-157,-0.02
Sun 11:00 AM,BAL @ BUF,4,Zay Flowers,BAL,+155,39.2,32.8,+205,-0.17


### Injuries the model is using
Official reports come out Wed-Fri, so the model also reads ESPN's live injury list (updates within hours of news) and treats it as the final word. A starting QB who's uncertain (Questionable/Doubtful) is simulated both ways, in proportion to his chance of playing. Check this table when a line looks off; override with `out=[...]` or `home_qb=` in section 2.

In [7]:
report['injuries']

,Team,Player,Pos,Status used,Chance out,Latest news
0,ARI,James Conner,RB,Injured Reserve (ESPN Oct 26: Surgery Foot),100%,When asked Monday if Conner (foot) will play t...
1,ARI,Jalen Brooks,WR,Injured Reserve (ESPN Oct 27: Surgery Wrist),100%,The Cardinals placed Brooks (wrist) on injured...
2,ARI,Tip Reiman,TE,Out (ESPN Oct 26: Not Specified Ankle),100%,"Head coach Mike LaFleur stated he's ""not too s..."
3,ARI,Andrew Wingard,S,Questionable (ESPN Oct 27: Bruise Knee),25%,questionable
4,ARI,Max Melton,CB,Questionable (ESPN Oct 27: Not Specified Knee),25%,questionable
5,ARI,Walter Nolen III,DT,Questionable (ESPN Oct 27: Not Specified Ankle),25%,questionable
6,ARI,Jon Gaines II,G,Questionable (ESPN Oct 27: Not Specified Knee),25%,questionable
7,ARI,Carson Beck,QB,Questionable (ESPN Oct 27: Not Specified Back),25%,questionable
8,ARI,Jeremiyah Love,RB,Questionable (ESPN Oct 26: Not Specified Ankle),25%,Head coach Mike LaFleur said Monday that Love ...
9,ATL,Olamide Zaccheaus,WR,Questionable (ESPN Oct 27: Not Specified Shoul...,25%,questionable


### Recent IR / Out moves
Everyone ESPN has put on IR, PUP, Out, or suspension since `SINCE` (None = last 4 days). `In model = True` means the simulations have him ruled out. If a move happened after your last run, re-run section 1 with `REFRESH_HOURS = 0`.

In [8]:
SINCE = None          # e.g. "2026-09-28"
ir_moves(report, since=SINCE)

ESPN injury list as of Wed Oct 28 08:13 AM Arizona time; 29 players listed since Oct 24.


,Player,Team,Pos,Status,Listed (AZ),Injury,In model
0,C.J. Gardner-Johnson,BUF,S,Suspension,Wed Oct 28 08:00 AM,Not Specified Suspension,True
1,Anthony Johnson Jr.,CHI,S,Injured Reserve,Wed Oct 28 07:59 AM,Undisclosed,True
2,Kalia Davis,CLE,DT,Injured Reserve,Tue Oct 27 05:26 PM,Not Specified Quadriceps,True
3,Cam Taylor-Britt,IND,CB,Injured Reserve,Tue Oct 27 04:40 PM,Not Specified Groin,True
4,Jalen Brooks,ARI,WR,Injured Reserve,Tue Oct 27 02:53 PM,Surgery Wrist,True
5,Sean Rhyan,GB,G,Out,Tue Oct 27 02:34 PM,Sprain Ankle,True
6,Tyrone Tracy Jr.,NYG,RB,Injured Reserve,Tue Oct 27 02:04 PM,Strain Hamstring,True
7,Erick All Jr.,CIN,TE,Injured Reserve,Tue Oct 27 11:33 AM,Surgery Knee - ACL,True
8,Taylor Moton,CAR,OT,Injured Reserve,Mon Oct 26 04:15 PM,Fracture Lower Leg,True
9,Jakorian Bennett,PHI,CB,Injured Reserve,Mon Oct 26 04:13 PM,Not Specified Knee,True


## 2. Explore one game with your own settings
`GAME_SIMS` sets how many times the game is played (every stat and prop is averaged across all of them). `home_qb` / `away_qb` force a starter, `out` rules players out ("C.Williams" or "Caleb Williams").

In [9]:
GAME_SIMS = 40000     # how many times to play this game (10,000-50,000; more = smoother %s, slower)

g = explore_game(report, home="GB", away="CAR",
                 home_qb=None, away_qb=None, out=[],
                 weather=None, referee='Alex kemp', n_sims=GAME_SIMS)
print(f"Simulated {len(g.game['home_scores']):,} games.")

CAR (B.Young) @ GB (J.Love)
  Win prob:    GB 60.4% | CAR 38.9% | tie 0.6%
  Final proj:  GB 25.4 - CAR 21.7   (halftime 13.7 - 9.8)
  Line:        GB -3.8   Total 47.1
  Possessions: GB 10.9, CAR 10.8   Turnovers: GB 1.19, CAR 1.31
  Most likely finals: GB 27 - CAR 20 (0.8%), GB 27 - CAR 17 (0.7%), GB 24 - CAR 17 (0.7%)
  Conditions:  ~55F, wind ~10 mph (stadium average, no forecast) (total -0.1) | rest: GB 4 days, CAR 4 days | CAR crosses 1 time zone(s) (GB -0.1)
  Referee:     Alex Kemp: 12.6 flags/game (league 12.1), away-minus-home +0.63 (121 games) (GB +0.00)
  GB offense: T.Smack (11/16, -0.17 vs avg) | QB pressured 15.1% of dropbacks (league 15.2%) | red-zone TD 58.2% (league 58.3%)
  CAR offense: R.Fitzgerald (36/41, +0.05 vs avg) | QB pressured 15.1% of dropbacks (league 15.2%) | red-zone TD 59.5% (league 58.3%)
  CAR defensive starters out: Jaycee Horn (CB, IR), Michael Jackson (CB, IR)
  GB defensive starters out: Zaire Franklin (LB, Doubtful (ESPN Oct 26: Fracture Shin)), 

In [10]:
position_table(g.props, "QB", stat="mean")

,player,team,pass_att,completions,pass_yds,pass_tds,rush_att,rush_yds,anytime_td,td_odds
0,Bryce Young,CAR,35.3,21.3,246.8,1.4,1.9,7.8,5%,+2023
1,Jordan Love,GB,34.1,20.4,233.6,2.0,1.2,5.3,4%,+2645


In [11]:
position_table(g.props, "RB", stat="mean")

,player,team,rush_att,rush_yds,targets,receptions,rec_yds,anytime_td,td_odds
0,Chuba Hubbard,CAR,15.1,64.9,2.6,2.0,14.3,53%,-111
1,AJ Dillon,CAR,3.6,13.1,0.6,0.4,2.9,9%,+957
2,Trevor Etienne,CAR,1.9,6.7,0.5,0.4,3.1,11%,+849
3,Josh Jacobs,GB,8.2,33.6,2.8,1.9,15.8,40%,+152
4,MarShawn Lloyd,GB,5.5,20.8,1.6,1.2,8.7,23%,+337
5,Pierre Strong,GB,5.0,19.5,3.9,2.9,21.5,30%,+238
6,Kaleb Johnson,GB,3.1,13.1,0.3,0.2,1.6,7%,+1242


In [12]:
position_table(g.props, "WR/TE", stat="mean")

,player,pos,team,targets,receptions,rec_yds,anytime_td,td_odds
0,Tetairoa McMillan,WR,CAR,7.1,4.6,64.4,22%,+345
1,Jalen Coker,WR,CAR,6.5,3.9,50.9,23%,+338
2,Darren Waller,TE,CAR,3.1,1.9,23.8,17%,+505
3,John Metchie III,WR,CAR,3.2,1.8,21.8,12%,+729
4,Tommy Tremble,TE,CAR,2.8,2.0,19.0,15%,+561
5,Ja'Tavion Sanders,TE,CAR,2.2,1.7,13.2,11%,+808
6,Christian Watson,WR,GB,5.7,3.2,53.3,34%,+197
7,Matthew Golden,WR,GB,5.5,2.8,42.6,26%,+282
8,Tucker Kraft,TE,GB,4.6,3.0,31.3,25%,+294
9,Drake Dabney,TE,GB,2.0,1.3,12.3,11%,+807


In [13]:
from nfl_weekly_v6 import _dk_edges

def game_edges(g, report):
    """Section 6's sportsbook table for the game in g, from g's own simulations."""
    if report.get('dk_ou') is None:
        print('No sportsbook lines in this report. Re-run Section 1.')
        return None
    h, a = g.props.home, g.props.away
    e = _dk_edges({(h, a): (g.game, g.props)}, report['dk_ou'], report['dk_yes'])
    if not len(e):
        print(f'No sportsbook lines posted yet for {a} @ {h}.')
        return e
    def _mean(r):
        if r.market == 'anytime_td':
            return float('nan')
        d = g.props.draws.get((r.player, r.market))
        return float(d.mean()) if d is not None else float('nan')
    e.insert(e.columns.get_loc('model_median'), 'model_mean', e.apply(_mean, axis=1).round(1))
    e = e.drop(columns='model_median')
    e.insert(2, 'book', report.get('lines_note', '').split(' ')[0])
    return e

print(report.get('lines_note', ''))
game_edges(g, report)

FanDuel lines and prices as of Wed Oct 28, 08:13 AM Arizona time (no attempts, completions or carries)


,player,game,book,market,line,model_mean,pick,model_prob,dk_price,fair_price,ev_per_unit
0,Pierre Strong,CAR @ GB,FanDuel,anytime_td,NaN,NaN,yes,0.295750,+1100,+238,2.549000
1,Josh Jacobs,CAR @ GB,FanDuel,rush_yds,58.5,33.6,under,0.848350,-114,-559,0.592523
2,Trevor Etienne,CAR @ GB,FanDuel,anytime_td,NaN,NaN,yes,0.105375,+1400,+849,0.580625
3,Matthew Golden,CAR @ GB,FanDuel,receptions,4.5,2.8,under,0.825475,-112,-473,0.562542
4,Tucker Kraft,CAR @ GB,FanDuel,receptions,4.5,3.0,under,0.803300,-112,-408,0.520567
5,AJ Dillon,CAR @ GB,FanDuel,anytime_td,NaN,NaN,yes,0.094650,+1500,+957,0.514400
6,Christian Watson,CAR @ GB,FanDuel,receptions,4.5,3.2,under,0.771800,-112,-338,0.460940
7,Tucker Kraft,CAR @ GB,FanDuel,rec_yds,46.5,31.3,under,0.772900,-114,-340,0.450888
8,Jalen Coker,CAR @ GB,FanDuel,receptions,4.5,3.9,under,0.653025,+120,-188,0.436655
9,Matthew Golden,CAR @ GB,FanDuel,rec_yds,59.5,42.6,under,0.738000,-114,-282,0.385374


## 3. One prop vs a line
`LINE = None` uses DraftKings' current line (needs the API key); type a number to test your own. Probabilities are counted straight from the simulations.

In [14]:
PLAYER, MARKET, LINE = "Garrett Wilson", "rec_yds", None
print(props.prop_check(g.props.medians, report["dk_ou"], report["dk_yes"], PLAYER, MARKET, line=LINE, sims=g.props))
g.props.distribution(PLAYER, MARKET)

Garrett Wilson isn't in this game's projections (check spelling or team).


## 4. Same-game combos
Legs: `(player, market, "over"/"under", line)`, `(player, "anytime_td")`, `(team, "win")`, `(team, "spread", -2.5)`, `("GAME", "total", "over", 44.5)`.

`correlation_lift` above 1 = the legs tend to hit together, so the combo is more likely than multiplying the single-leg odds. Put `dk_price="+450"` after the closing `]` (not inside the list) to compare with DraftKings' same-game-parlay price (the Odds API doesn't provide SGP prices, so type it in).

## 5. Most likely TD scorer per game
Shortest fair anytime-TD odds in each game, straight from the simulations. `TOP = 3` lists three per game. DraftKings price and EV per $1 show up when lines were pulled.

In [15]:
TOP = 1
favs = td_favorites(report, top=TOP)
favs.to_csv(f"reports/{SEASON}_wk{WEEK:02d}_td_favorites.csv", index=False)
favs

,Game,Kickoff (AZ),Player,Pos,Team,Model TD %,Fair odds,Expected TDs,Book price,EV per $1
0,CAR @ GB,Thu 05:15 PM,Chuba Hubbard,RB,CAR,52.7,-111,0.70,-115,-0.015
1,BAL @ BUF,Sun 11:00 AM,Derrick Henry,RB,BAL,73.0,-270,1.15,-240,0.034
2,TEN @ CIN,Sun 11:00 AM,Chase Brown,RB,CIN,44.8,+123,0.56,NaN,NaN
3,ARI @ DAL,Sun 11:00 AM,Javonte Williams,RB,DAL,69.7,-230,1.08,NaN,NaN
4,MIN @ DET,Sun 11:00 AM,Jahmyr Gibbs,RB,DET,60.0,-150,0.83,-290,-0.193
5,IND @ JAX,Sun 11:00 AM,Jonathan Taylor,RB,IND,55.1,-123,0.73,NaN,NaN
6,LV @ NYJ,Sun 11:00 AM,Ashton Jeanty,RB,LV,49.5,+102,0.64,NaN,NaN
7,CLE @ PIT,Sun 11:00 AM,Jaylen Warren,RB,PIT,39.4,+154,0.48,NaN,NaN
8,ATL @ TB,Sun 11:00 AM,Bijan Robinson,RB,ATL,56.2,-128,0.76,NaN,NaN
9,LAC @ LA,Sun 02:05 PM,Kyren Williams,RB,LA,54.8,-121,0.73,NaN,NaN


## 6. Sportsbook props vs the model
Sorted by expected value per $1. The `book` column says where each line came from. Sources (`LINES` in section 1), all but the last free with no key:

| Source | Prices | Anytime TD | Attempts / completions / carries |
|---|---|---|---|
| **FanDuel** (`"fanduel"`) | Real | Yes | No |
| **ESPN** (`"espn"`, DraftKings' lines) | -110 assumed | No | Yes |
| **The Odds API** (`"odds_api"`, key needed) | Real DraftKings | Yes | Yes |

`"auto"` uses The Odds API if `odds_api_key.txt` exists, then FanDuel, then ESPN. Books post Sunday props by about Wednesday.

Huge edges usually mean the model is missing something (a new starter, a role change), not that the book is wrong.

In [16]:
e = report['edges'].copy()
if len(e):
    def _mean(r):
        if r.market == 'anytime_td':
            return float('nan')
        a, h = r.game.split(' @ ')
        d = report['sims'][(h, a)][1].draws.get((r.player, r.market))
        return float(d.mean()) if d is not None else float('nan')
    e.insert(e.columns.get_loc('model_median'), 'model_mean', e.apply(_mean, axis=1).round(1))
    e = e.drop(columns='model_median')
print(report.get('lines_note', ''))
e.head(40) if len(e) else print('No sportsbook lines found yet for this week.')

FanDuel lines and prices as of Wed Oct 28, 08:13 AM Arizona time (no attempts, completions or carries)


,player,game,book,market,line,model_mean,pick,model_prob,dk_price,fair_price,ev_per_unit
0,Pierre Strong,CAR @ GB,FanDuel,anytime_td,NaN,NaN,yes,0.295750,+1100,+238,2.549000
1,Kaytron Allen,NE @ MIA,FanDuel,anytime_td,NaN,NaN,yes,0.157450,+2000,+535,2.306450
2,Ty Johnson,BAL @ BUF,FanDuel,anytime_td,NaN,NaN,yes,0.319200,+650,+213,1.394000
3,DJ Moore,BAL @ BUF,FanDuel,rec_yds,54.5,24.0,under,0.894300,-114,-846,0.678780
4,Josh Jacobs,CAR @ GB,FanDuel,rush_yds,58.5,33.6,under,0.848350,-114,-559,0.592523
5,Trevor Etienne,CAR @ GB,FanDuel,anytime_td,NaN,NaN,yes,0.105375,+1400,+849,0.580625
6,Kevin Coleman Jr.,NE @ MIA,FanDuel,anytime_td,NaN,NaN,yes,0.121175,+1200,+725,0.575275
7,DJ Moore,BAL @ BUF,FanDuel,receptions,4.5,1.8,under,0.953200,-154,-2037,0.572208
8,Matthew Golden,CAR @ GB,FanDuel,receptions,4.5,2.8,under,0.825475,-112,-473,0.562542
9,Justin Jefferson,MIN @ DET,FanDuel,receptions,5.5,4.3,under,0.725275,+112,-264,0.537583


In [17]:
def td_board(report, top=4):
    """Each game's top `top` anytime-TD favorites by sportsbook odds, with the model's take and EV."""
    dk = report.get("dk_yes")
    if dk is None or not len(dk):
        print("No anytime-TD prices in this report. Set LINES = 'fanduel' (or 'auto') in Section 1 and re-run it.")
        return None
    kick = report["games"].set_index("matchup").kickoff_utc
    rows = []
    for (h, a), (g, sp) in report["sims"].items():
        game = f"{a} @ {h}"
        teams = sp.means.set_index("player").team
        d = dk[(dk.home == h) & (dk.away == a)].sort_values("yes_price").drop_duplicates("key")
        n = 0
        for _, r in d.iterrows():
            name = sp._name(r.player)
            if (name, "anytime_td") not in sp.draws:
                continue          # team defenses / players the model doesn't project
            p = sp.prob_over(name, "anytime_td", None)
            n += 1
            k = kick.get(game)
            rows.append({"_kick": k, "Kickoff (AZ)": k.tz_convert("America/Phoenix").strftime("%a %I:%M %p"),
                         "Game": game, "Rank": n, "Player": name, "Team": teams.get(name, ""),
                         "Book odds": props._american(r.yes_price),
                         "Book %": round(100 / r.yes_price, 1),
                         "Model %": round(p * 100, 1),
                         "Model fair odds": props._american_from_prob(p),
                         "EV per $1": round(p * r.yes_price - 1, 3)})
            if n == top:
                break
    if not rows:
        print("No anytime-TD prices matched this week's games yet.")
        return None
    return pd.DataFrame(rows).sort_values(["_kick", "Game", "Rank"]).drop(columns="_kick").reset_index(drop=True)

TOP = 4
board = td_board(report, top=TOP)
if board is not None:
    board.to_csv(f"reports/{SEASON}_wk{WEEK:02d}_td_board.csv", index=False)
    display(board.style
            .format({"Book %": "{:.1f}", "Model %": "{:.1f}", "EV per $1": "{:+.2f}"})
            .background_gradient(subset=["EV per $1"], cmap="RdYlGn", vmin=-0.4, vmax=0.4)
            .hide(axis="index"))

Kickoff (AZ),Game,Rank,Player,Team,Book odds,Book %,Model %,Model fair odds,EV per $1
Thu 05:15 PM,CAR @ GB,1,Josh Jacobs,GB,-135,57.4,39.7,+152,-0.31
Thu 05:15 PM,CAR @ GB,2,Chuba Hubbard,CAR,-115,53.5,52.7,-111,-0.01
Thu 05:15 PM,CAR @ GB,3,Christian Watson,GB,+130,43.5,33.7,+197,-0.23
Thu 05:15 PM,CAR @ GB,4,Matthew Golden,GB,+160,38.5,26.2,+282,-0.32
Sun 11:00 AM,BAL @ BUF,1,Derrick Henry,BAL,-240,70.6,73.0,-270,+0.03
Sun 11:00 AM,BAL @ BUF,2,Josh Allen,BUF,-170,63.0,38.4,+160,-0.39
Sun 11:00 AM,BAL @ BUF,3,James Cook,BUF,-165,62.3,61.1,-157,-0.02
Sun 11:00 AM,BAL @ BUF,4,Zay Flowers,BAL,+155,39.2,32.8,+205,-0.17
Sun 11:00 AM,MIN @ DET,1,Jahmyr Gibbs,DET,-290,74.4,60.0,-150,-0.19
Sun 11:00 AM,MIN @ DET,2,Aaron Jones,MIN,+100,50.0,45.0,+122,-0.10


## 7. Track record
Grades every logged prediction whose game has finished. Watch **closing line value** first: it becomes informative after a few dozen games, while win-loss records need hundreds.

In [18]:
track = grade_log()

Games: {'games graded': 59, 'winner %': '59.3%', 'spread miss (model)': np.float64(8.5), 'spread miss (market at log)': np.float64(8.6), 'ATS vs logged line': '30-24 (55.6%)', 'spread CLV (avg pts)': np.float64(0.03), 'beat the closing spread': '8.5% (moved away: 10.2%)', 'O/U vs logged total': '23-36 (39.0%)', 'total CLV (avg pts)': np.float64(0.12)}
Props: {'props graded': 1927, 'record': '856-1071', 'profit per $1': np.float64(-0.034), 'line CLV (avg)': np.float64(0.69), 'beat the closing line': '1.7%'}
Only 59 games graded: records this small are mostly luck. CLV becomes informative first.


## 8. Reference tables

In [19]:
ns.coach_table(report['model'])

,coach,4th_GOE_punt_zone,4th_GOE_fg_zone,halftime_off_swing,halftime_def_swing
team,,,,,
DAL,Brian Schottenheimer,0.129,0.045,-0.017,0.028
IND,Shane Steichen,0.083,-0.007,-0.027,-0.033
DET,Dan Campbell,0.068,0.186,0.007,-0.037
BUF,Sean McDermott,0.064,0.148,0.021,0.042
PHI,Nick Sirianni,0.057,0.070,-0.083,-0.028
CAR,Dave Canales,0.056,-0.006,0.033,0.035
ATL,Raheem Morris,0.048,0.002,0.001,-0.016
KC,Andy Reid,0.046,-0.082,0.001,0.007
NYG,John Harbaugh,0.025,-0.000,0.052,-0.038


In [20]:
ns.qb_table(report['model']).head(32)

,name,team,db,epa_per_db,turnover_pct
qb_id,,,,,
00-0033873,P.Mahomes,KC,5549,0.212,1.877
00-0037834,B.Purdy,SF,2004,0.192,2.463
00-0034857,J.Allen,BUF,5101,0.188,2.220
00-0034796,L.Jackson,BAL,3603,0.171,2.169
00-0036442,J.Burrow,CIN,3658,0.135,2.101
00-0033077,D.Prescott,DAL,4082,0.132,2.280
00-0033106,J.Goff,DET,4718,0.120,2.163
00-0026498,M.Stafford,LA,4300,0.116,2.257
00-0039910,J.Daniels,WAS,1122,0.114,1.661


In [21]:
ns.active_referees(report['model'])

,games,flags_pg,away_minus_home_flags
referee,,,
Adrian Hill,118,12.75,0.60
Alan Eck,54,11.25,0.11
Alex Kemp,121,12.58,0.63
Alex Moore,21,14.24,0.79
Bill Vinovich,122,11.09,0.28
Brad Allen,118,11.67,0.56
Brad Rogers,112,12.38,0.11
Carl Cheffers,121,12.49,0.54
Clay Martin,117,11.87,0.42


## 9. Game backtest vs Vegas (optional, ~3 minutes per season)

In [22]:
RUN_BACKTEST = True
if RUN_BACKTEST:
    D = report["data"]
    bt, _ = ns.backtest(D["games"], D["prep"], D["injuries"], ns.load_rosters(2025)[0], 2025, n_sims=4000)
    display(ns.grade_vs_vegas(bt))

games                      224
winner_pct_model      0.623318
winner_pct_vegas      0.632287
spread_miss_model     10.22375
spread_miss_vegas     9.924107
total_miss_model     10.230971
total_miss_vegas        9.9375
ATS_record             111-112
ATS_pct               0.497758
ATS_p_value           0.553251
OU_record              118-106
OU_pct                0.526786
OU_p_value            0.231218
model_total_bias      1.108663
dtype: object

In [23]:
e = report['edges'][report['edges'].market != 'anytime_td']
(e.assign(gap=e.model_median - e.line)
  .groupby('market').gap.agg(avg_gap='mean', props='count').round(1))

,avg_gap,props
market,,
pass_tds,-0.1,13
pass_yds,-11.7,13
rec_yds,-6.3,55
receptions,-0.7,55
rush_yds,-5.1,28
